In [49]:
import numpy as np
import  pandas as pd
import warnings

warnings.filterwarnings('ignore')


In [50]:
df = pd.read_csv('qoute_dataset.csv')

In [51]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [52]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [53]:
quotes[0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [54]:
quotes = quotes.str.lower()

In [55]:
quotes[0]

'“the world as we have created it is a process of our thinking. it cannot be changed without changing our thinking.”'

In [56]:
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x : x.translate(translator))

In [57]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [58]:
from tensorflow.keras.preprocessing.text import Tokenizer
vocab_size = 10000

tokenizer = Tokenizer(num_words= vocab_size)
tokenizer.fit_on_texts(quotes)

In [59]:
word_index = tokenizer.word_index
print(len(word_index))

8978


In [60]:
sentences = tokenizer.texts_to_sequences(quotes)

In [61]:
X = []
y = []

for seq in sentences:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)


In [62]:
len(X)

85271

In [63]:
len(y)

85271

In [64]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [65]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_pad = pad_sequences(X, maxlen = max_len, padding= 'pre')

In [66]:
X_pad

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]],
      shape=(85271, 745), dtype=int32)

In [67]:
y = np.array(y)

In [68]:
X_pad.shape

(85271, 745)

In [70]:
from tensorflow.keras.utils import to_categorical
y_onehot = to_categorical(y, num_classes= vocab_size )

In [71]:
y_onehot.shape

(85271, 10000)

In [72]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,SimpleRNN

In [73]:
embedded_dim = 50
rnn_units = 128


In [74]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedded_dim, input_length=max_len))
rnn_model.add(SimpleRNN(units = rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))


W0000 00:00:1786186528.128324    3400 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [75]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [76]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [80]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedded_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [81]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [83]:
lstm_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [84]:
epochs = 10
batch_size = 128


In [ ]:
hist_rnn = rnn_model.fit(
    X_pad, y_onehot, epochs=epochs, batch_size=batch_size, verbose=1,validation_split=0.2
)